<a href="https://colab.research.google.com/github/serracc/Philab/blob/main/Serrbi_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Remove conflicting preinstalled packages in Colab
!pip uninstall -y pydrive2 mcp

# Install project deps and Playwright
!pip install -q Crawl4AI==0.4.247 pydantic==2.10.6 python-dotenv==1.0.1 playwright nest_asyncio

# Install Playwright browsers/deps (Chromium is enough for this project)
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entr

In [ ]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"

os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"

In [ ]:
# Cell 3 - Config (DreamJob scraping setup)

LLM_MODEL = "openrouter/meta-llama/llama-3.1-8b-instruct"
API_TOKEN = os.getenv("OPENROUTER_API_KEY")

BASE_URL = "https://www.dreamjob.ma/emploi/casablanca/"
MAX_PAGES = 4  # Scrape 4 pages for now

# DreamJob CSS selector for job cards
CSS_SELECTOR = "article.jeg_post"

SCRAPER_INSTRUCTIONS = (
    "You are an expert data extractor. "
    "From each job listing card, extract: title, company name, and link to job details. "
    "Then, visit each job link and extract: the full job description, any email found, and any application link found. "
    "Infer category, experience level, type, location requirement, and tags based on description content. "
    "Categories: Tech, Finance, Hospitality, Health, Legal, Construction, Education, CallCenter, Auto, Cleaning, Other. "
    "ExperienceLevel: junior, mid_level, senior. "
    "Type: internship, part_time, full_time. "
    "LocationRequirement: in_office, hybrid, remote. "
    "Tags depend on category: "
    "Tech=[react,nextjs,typescript,javascript,node,express,python,django,java,spring,aws,docker,kubernetes,graphql,postgres,mongodb,ml,ai,pytorch,tensorflow,devops,ci,cd,tailwind,vite], "
    "Finance=[accounting,audit,excel,fp&a,budgeting,tax,reconciliation], "
    "Hospitality=[front desk,customer service,barista,housekeeping,waiter,chef], "
    "Construction=[carpentry,plumbing,masonry,electrical,hvac,blueprints], "
    "Education=[teaching,curriculum,lesson plan,assessment,tutoring], "
    "Legal=[contract,litigation,compliance,due diligence], "
    "Health=[patient care,nursing,clinic,medical records], "
    "Cleaning=[deep clean,janitorial,sanitation,disinfection], "
    "Auto=[mechanic,diagnostics,oil change,brakes], Other=[]. "
    "Ensure each extracted job includes title, companyName, description, applicationEmail, applicationUrl, city='Casablanca', and reasonable defaults for missing data."
)


In [ ]:
# Cell 4 - models/job.py (Pydantic schema for jobs)

from pydantic import BaseModel, Field
from typing import List, Optional

class JobData(BaseModel):
    id: str = Field(..., description="Unique UUID for job")
    userId: str = Field(..., description="User ID who added the job")
    title: str
    companyName: Optional[str] = None
    companyImage: Optional[str] = None
    description: str
    wage: Optional[int] = None
    stateAbbreviation: Optional[str] = "CAS"
    city: str = "Casablanca"
    category: str
    locationRequirement: str
    experienceLevel: str
    status: str = "pending"
    type: str
    applicationEmail: str
    applicationUrl: Optional[str] = None
    postedAt: Optional[str] = None
    rejectionReason: Optional[str] = None
    moderatedAt: Optional[str] = None
    moderatedBy: Optional[str] = None
    createdAt: str
    updatedAt: str
    tags: List[str] = []


In [ ]:
# Cell 5 - src/utils.py

import csv, json, uuid
from datetime import datetime

def generate_uuid() -> str:
    return str(uuid.uuid4())

def save_data(records: list, filename_csv: str, filename_json: str):
    if not records:
        print("⚠️ No records to save.")
        return

    # Save CSV
    keys = records[0].keys()
    with open(filename_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(records)
    print(f"💾 Saved {len(records)} jobs to CSV → {filename_csv}")

    # Save JSON
    with open(filename_json, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved JSON copy → {filename_json}")


In [ ]:
# Cell 6 - src/scraper.py

import json, re, asyncio
from bs4 import BeautifulSoup
from crawl4ai import (
    AsyncWebCrawler,
    BrowserConfig,
    CacheMode,
    CrawlerRunConfig,
    LLMExtractionStrategy,
)
from datetime import datetime
from typing import List, Set

from uuid import uuid4

from pydantic import BaseModel

def get_browser_config():
    return BrowserConfig(
        browser_type="chromium",
        headless=True,
        verbose=False,
    )

def get_llm_strategy(instructions: str, output_format: BaseModel):
    return LLMExtractionStrategy(
        provider=LLM_MODEL,
        api_token=API_TOKEN,
        schema=output_format.model_json_schema(),
        extraction_type="schema",
        instruction=instructions,
        input_format="markdown",
        verbose=False,
    )

async def extract_job_links(crawler, url):
    result = await crawler.arun(url=url, config=CrawlerRunConfig(cache_mode=CacheMode.BYPASS))
    soup = BeautifulSoup(result.html, "html.parser")
    articles = soup.select("article.jeg_post")
    jobs = []
    for article in articles:
        a = article.select_one(".jeg_post_title a")
        if a:
            jobs.append({
                "title": a.get_text(strip=True),
                "link": a["href"]
            })
    return jobs

async def extract_job_detail(crawler, job_link, llm_strategy):
    result = await crawler.arun(
        url=job_link,
        config=CrawlerRunConfig(
            cache_mode=CacheMode.BYPASS,
            extraction_strategy=llm_strategy,
        ),
    )
    if not (result.success and result.extracted_content):
        return None
    return json.loads(result.extracted_content)


In [ ]:
# ✅ Fixed Cell 7 — handles list results from LLM properly

import nest_asyncio, asyncio, time
from datetime import datetime, timezone
from uuid import uuid4

nest_asyncio.apply()

USER_ID = "e8d67fde-c8d4-4fe1-b17a-749ebc0afb50"

async def crawl_dreamjob():
    browser_config = get_browser_config()
    llm_strategy = get_llm_strategy(SCRAPER_INSTRUCTIONS, JobData)
    session_id = "dreamjob_session"

    all_jobs = []
    seen_links = set()

    async with AsyncWebCrawler(config=browser_config) as crawler:
        for page in range(1, MAX_PAGES + 1):
            url = BASE_URL if page == 1 else f"{BASE_URL}page/{page}/"
            print(f"🔎 Crawling page {page}: {url}")
            jobs = await extract_job_links(crawler, url)

            for j in jobs:
                if j["link"] in seen_links:
                    continue
                seen_links.add(j["link"])

                print(f"→ Visiting job: {j['title']}")
                detail = await extract_job_detail(crawler, j["link"], llm_strategy)
                if not detail:
                    continue

                # 🩹 Fix for list vs dict
                if isinstance(detail, list):
                    if len(detail) == 0:
                        continue
                    detail = detail[0]

                if not isinstance(detail, dict):
                    print(f"⚠️ Skipped invalid detail format for {j['title']}")
                    continue

                now = datetime.now(timezone.utc).isoformat()
                job = {
                    "id": str(uuid4()),
                    "userId": USER_ID,
                    "createdAt": now,
                    "updatedAt": now,
                    **detail,
                }
                all_jobs.append(job)
                await asyncio.sleep(1)  # polite delay

    if all_jobs:
        save_data(all_jobs, "jobs_dreamjob.csv", "jobs_dreamjob.json")
        print(f"🎉 Done! {len(all_jobs)} jobs saved.")
    else:
        print("⚠️ No jobs scraped.")

# --- Run it ---
try:
    asyncio.run(crawl_dreamjob())
except KeyboardInterrupt:
    print("🛑 Stopped manually")
except Exception as e:
    print(f"❌ Error: {e}")


🔎 Crawling page 1: https://www.dreamjob.ma/emploi/casablanca/
→ Visiting job: Infirmiers d’entreprise recherchés chez SCIF – Postulez dès maintenant
→ Visiting job: Robobat Maroc recrute à Casablanca – Postes urgents à pourvoir dès maintenant !
→ Visiting job: CTR Maroc recrute – Postes de techniciens à pourvoir dès maintenant !
→ Visiting job: Deloitte Maroc lance sa campagne de stages Audit PFE 2026
→ Visiting job: Audit, paie, comptabilité: ECOVIS Morocco recrute à Casablanca
→ Visiting job: Emploi santé 2025 – AKDITAL ouvre de nombreux postes à Casablanca
→ Visiting job: Mamda et Mcma recrutent des stagiaires rémunérés dans tout le Maroc
→ Visiting job: Atlas Multiservices recrute des Agents Polyvalents (98 Postes)
→ Visiting job: Emploi BTP 2025 – G3C recherche des profils techniques expérimentés
→ Visiting job: Ma-Navette recrute à Casablanca – Agents d’accueil et de régulation demandés
→ Visiting job: Kenzi Tower offre des stages en RH, cuisine et restauration
→ Visiting job: Un

In [ ]:
from google.colab import files
files.download("jobs_dreamjob.csv")
